# Column transformer
Apply a different transformation to each column of the COVID toy data, first by hand, then with scikit-learn's `ColumnTransformer`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

## 1. The data
100 made-up patients: age, gender, fever, cough, city, and whether they have COVID.

In [ ]:
# load the toy dataset
df = pd.read_csv("data/covid_toy.csv")
df.head()

In [ ]:
# how many patients have a mild or a strong cough
df["cough"].value_counts()

In [ ]:
# the four cities
df["city"].value_counts()

In [ ]:
# count missing values per column: only fever has some
df.isnull().sum()

## 2. Split before transforming

In [ ]:
# inputs = every column except the target; 20% of rows go to the test set
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["has_covid"]), df["has_covid"],
    test_size=0.2, random_state=0)
X_train.shape, X_test.shape

## 3. The hard way: one transformer at a time
Each transformer is fitted on the training set only, then applied to both sets.

In [ ]:
# fever: fill missing values with the training mean
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.transform(X_test[["fever"]])
X_train_fever.shape

In [ ]:
# cough: ordinal encoding, Mild = 0, Strong = 1
oe = OrdinalEncoder(categories=[["Mild", "Strong"]])
X_train_cough = oe.fit_transform(X_train[["cough"]])
X_test_cough = oe.transform(X_test[["cough"]])
X_train_cough.shape

In [ ]:
# gender and city: one-hot encoding, dropping the first category of each
# 1 column for gender + 3 for city = 4 columns
ohe = OneHotEncoder(drop="first", sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]])
X_test_gender_city = ohe.transform(X_test[["gender", "city"]])
X_train_gender_city.shape

In [ ]:
# age: needs nothing, so keep it as a NumPy array
X_train_age = X_train[["age"]].values
X_test_age = X_test[["age"]].values
X_train_age.shape

In [ ]:
# glue the four pieces side by side (axis=1 means column-wise)
X_train_transformed = np.concatenate(
    (X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate(
    (X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)
X_train_transformed.shape, X_test_transformed.shape

## 4. The easy way: ColumnTransformer

In [ ]:
# each tuple: (name, transformer, list of columns)
# remainder="passthrough" keeps the untouched column (age)
transformer = ColumnTransformer(transformers=[
    ("tnf1", SimpleImputer(), ["fever"]),
    ("tnf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
    ("tnf3", OneHotEncoder(sparse_output=False, drop="first"), ["gender", "city"]),
], remainder="passthrough")

In [ ]:
# learn from the training set and transform it, in one call
X_train_ct = transformer.fit_transform(X_train)
X_train_ct.shape

In [ ]:
# apply what was learned to the test set (no fitting)
X_test_ct = transformer.transform(X_test)
X_test_ct.shape

In [ ]:
# the first three training rows before ...
X_train.head(3)

In [ ]:
# ... and after: fever, cough, gender_Male, city_Delhi, city_Kolkata, city_Mumbai, age
X_train_ct[:3]

## 5. Which output column is which

In [ ]:
# names of the output columns, in order (prefix = transformer name)
transformer.get_feature_names_out()

In [ ]:
# ask for a DataFrame with those names instead of a bare array
transformer.set_output(transform="pandas")
transformer.fit_transform(X_train).head()

In [ ]:
# the same columns as the hand-made array, only in a different order
X_hand = pd.DataFrame(X_train_transformed,
    columns=["age", "fever", "gender_Male", "city_Delhi",
             "city_Kolkata", "city_Mumbai", "cough"])
X_ct = transformer.transform(X_train)
X_ct.columns = [c.split("__")[1] for c in X_ct.columns]
np.allclose(X_hand[X_ct.columns].values, X_ct.values)

In [ ]:
# each fitted transformer is kept inside, under its name
transformer.named_transformers_["tnf1"].statistics_  # training mean of fever

## 6. remainder='drop'

In [ ]:
# the default: columns without a transformer are thrown away (age disappears)
dropper = ColumnTransformer(transformers=[
    ("tnf1", SimpleImputer(), ["fever"]),
    ("tnf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
    ("tnf3", OneHotEncoder(sparse_output=False, drop="first"), ["gender", "city"]),
])
dropper.fit_transform(X_train).shape